In [1]:
import os
for i in range(1, 12):
    os.makedirs(f"{i:02d}", exist_ok=True)

# A1

In [2]:
import gmsh

gmsh.initialize()

gmsh.model.add("glacier_terminus")

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 500.0
Ly = 750.0
Lz = 125.0

# Pre-crack / notch dimensions [m]
lx = 5.0
ly = 10.0
lz = 10.0

# -------------------------------------------------------------------------
# Mesh size controls [m]
# -------------------------------------------------------------------------

h_min = 2.0
h_max = 6.15

# Refined zone size
h_refined = 2.5

gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)

# -------------------------------------------------------------------------
# Glacier geometry
# -------------------------------------------------------------------------

glacier = gmsh.model.occ.addBox(
    0.0,
    0.0,
    0.0,
    Lx,
    Ly,
    Lz,
)

# -------------------------------------------------------------------------
# Notch
# -------------------------------------------------------------------------

notch_x0 = 0.5 * Lx - 0.5 * lx
notch_y0 = 0.0
notch_z0 = Lz - lz

notch = gmsh.model.occ.addBox(
    notch_x0,
    notch_y0,
    notch_z0,
    lx,
    ly,
    lz,
)

# Subtract notch from glacier
domain, _ = gmsh.model.occ.cut(
    [(3, glacier)],
    [(3, notch)],
    removeObject=True,
    removeTool=True,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Internal cutting planes
#
# These force conforming mesh surfaces and therefore nodes at:
#
# z = Lz / 4 = 31.25 m
# z = Lz / 2 = 62.50 m
# -------------------------------------------------------------------------

z_quarter = Lz / 4.0
z_half = Lz / 2.0

plane_quarter = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    z_quarter,
    Lx,
    Ly,
)

plane_half = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    z_half,
    Lx,
    Ly,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Fragment the glacier with the two internal planes
# -------------------------------------------------------------------------

domain, _ = gmsh.model.occ.fragment(
    domain,
    [
        (2, plane_quarter),
        (2, plane_half),
    ],
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Physical volume
# -------------------------------------------------------------------------

volume_tags = [tag for dim, tag in domain if dim == 3]

gmsh.model.addPhysicalGroup(
    3,
    volume_tags,
    1,
)

gmsh.model.setPhysicalName(
    3,
    1,
    "GLACIER",
)

# -------------------------------------------------------------------------
# Local refinement zone
#
# Crack center:
#       x = 250 m
#
# Refined region:
#       x = 220 -> 280 m
#       y =   0 -> 750 m
#       z =   0 -> 125 m
#
# -------------------------------------------------------------------------

crack_center_x = 0.5 * Lx

refinement_width = 30.0

field_box = gmsh.model.mesh.field.add("Box")

gmsh.model.mesh.field.setNumber(
    field_box,
    "VIn",
    h_refined,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "VOut",
    h_max,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "XMin",
    crack_center_x - refinement_width,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "XMax",
    crack_center_x + refinement_width,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "YMin",
    0.0,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "YMax",
    Ly,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "ZMin",
    0.0,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "ZMax",
    Lz,
)

# -------------------------------------------------------------------------
# Smooth transition around refined zone
# -------------------------------------------------------------------------

gmsh.model.mesh.field.setNumber(
    field_box,
    "Thickness",
    10.0,
)

gmsh.model.mesh.field.setAsBackgroundMesh(
    field_box
)

# -------------------------------------------------------------------------
# Mesh settings
# -------------------------------------------------------------------------

gmsh.option.setNumber(
    "Mesh.Algorithm3D",
    10,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeFromCurvature",
    0,
)

# Prevent point/curve/surface sizes from overriding background field
gmsh.option.setNumber(
    "Mesh.MeshSizeFromPoints",
    0,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeExtendFromBoundary",
    0,
)

# -------------------------------------------------------------------------
# Generate tetrahedral mesh
# -------------------------------------------------------------------------

gmsh.model.mesh.generate(3)


gmsh.write("01/Lx500_1C_NA.msh")

gmsh.finalize()

Info    : Meshing 1D...                                                                                                                 
Info    : [  0%] Meshing curve 38 (Line)
Info    : [ 10%] Meshing curve 39 (Line)
Info    : [ 10%] Meshing curve 40 (Line)
Info    : [ 10%] Meshing curve 41 (Line)
Info    : [ 20%] Meshing curve 42 (Line)
Info    : [ 20%] Meshing curve 43 (Line)
Info    : [ 20%] Meshing curve 44 (Line)
Info    : [ 20%] Meshing curve 45 (Line)
Info    : [ 30%] Meshing curve 46 (Line)
Info    : [ 30%] Meshing curve 47 (Line)
Info    : [ 30%] Meshing curve 48 (Line)
Info    : [ 30%] Meshing curve 49 (Line)
Info    : [ 40%] Meshing curve 50 (Line)
Info    : [ 40%] Meshing curve 51 (Line)
Info    : [ 40%] Meshing curve 52 (Line)
Info    : [ 40%] Meshing curve 53 (Line)
Info    : [ 50%] Meshing curve 54 (Line)
Info    : [ 50%] Meshing curve 55 (Line)
Info    : [ 50%] Meshing curve 56 (Line)
Info    : [ 50%] Meshing curve 57 (Line)
Info    : [ 60%] Meshing curve 58 (Line)
In

In [3]:
import meshio

mesh = meshio.read("01/Lx500_1C_NA.msh")

cells = mesh.get_cells_type("tetra")
points = mesh.points

meshio.write("01/Lx500_1C_NA.xdmf", meshio.Mesh(
    points=points,
    cells={"tetra": cells}))

# A2

In [5]:
import gmsh


gmsh.initialize()
gmsh.model.add("glacier_terminus")

# =========================================================================
# Geometry dimensions [m]
# =========================================================================

Lx = 500.0
Ly = 750.0
Lz = 125.0

# Pre-crack / notch dimensions [m]
lx = 5.0
ly = 10.0
lz = 10.0

# =========================================================================
# Mesh size controls [m]
# =========================================================================

h_min = 1.0
h_max = 20.0

gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)
gmsh.option.setNumber("Mesh.MeshSizeFactor", 1.0)

# Prevent automatic size estimates from changing the background field.
gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)

# =========================================================================
# Glacier geometry
# =========================================================================

glacier = gmsh.model.occ.addBox(
    0.0,
    0.0,
    0.0,
    Lx,
    Ly,
    Lz,
)

# =========================================================================
# Notch geometry
# =========================================================================

notch_x0 = 0.5 * Lx - 0.5 * lx
notch_y0 = 0.0
notch_z0 = Lz - lz

notch = gmsh.model.occ.addBox(
    notch_x0,
    notch_y0,
    notch_z0,
    lx,
    ly,
    lz,
)

# Subtract notch from glacier
domain, _ = gmsh.model.occ.cut(
    [(3, glacier)],
    [(3, notch)],
    removeObject=True,
    removeTool=True,
)

gmsh.model.occ.synchronize()

# =========================================================================
# Internal cutting planes
#
# z = Lz/4
# z = Lz/2
#
# These guarantee mesh nodes at these elevations.
# =========================================================================

z1 = Lz / 4.0
z2 = Lz / 2.0

plane_z1 = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    z1,
    Lx,
    Ly,
)

plane_z2 = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    z2,
    Lx,
    Ly,
)

gmsh.model.occ.synchronize()

# =========================================================================
# Fragment glacier with internal planes
# =========================================================================

domain, _ = gmsh.model.occ.fragment(
    domain,
    [
        (2, plane_z1),
        (2, plane_z2),
    ],
)

gmsh.model.occ.synchronize()

# =========================================================================
# Physical volume
# =========================================================================

volume_tags = [
    tag
    for dim, tag in domain
    if dim == 3
]

gmsh.model.addPhysicalGroup(
    3,
    volume_tags,
    1,
)

gmsh.model.setPhysicalName(
    3,
    1,
    "GLACIER",
)

# =========================================================================
# Identify notch/crack surfaces
# =========================================================================

tol = 1e-6

crack_surfaces = []

for dim, tag in gmsh.model.getEntities(2):

    xmin, ymin, zmin, xmax, ymax, zmax = (
        gmsh.model.getBoundingBox(dim, tag)
    )

    # Surfaces associated with the notch:
    #
    #   x within notch width
    #   y near front
    #   z near top
    #
    if (
        xmin >= notch_x0 - tol
        and xmax <= notch_x0 + lx + tol
        and ymin <= ly + tol
        and zmax >= notch_z0 - tol
    ):
        crack_surfaces.append(tag)

print("Crack surface tags:", crack_surfaces)

# =========================================================================
# Local 3D refinement using a Box field
# =========================================================================
#
# Fine mesh:
#     VIn = 1 m
#
# Coarse mesh:
#     VOut = 20 m
#
# The fine region is deliberately small to target ~O(10^4) vertices
# rather than the ~40k vertices produced by the previous large box.
# =========================================================================

box_field = gmsh.model.mesh.field.add("Box")

gmsh.model.mesh.field.setNumber(
    box_field,
    "VIn",
    h_min,
)

gmsh.model.mesh.field.setNumber(
    box_field,
    "VOut",
    h_max,
)

# -------------------------------------------------------------------------
# Refinement box around notch
# -------------------------------------------------------------------------

box_margin_x = 3.0
box_margin_y = 5.0
box_margin_z = 5.0

box_xmin = max(
    0.0,
    notch_x0 - box_margin_x,
)

box_xmax = min(
    Lx,
    notch_x0 + lx + box_margin_x,
)

box_ymin = 0.0

box_ymax = min(
    Ly,
    ly + box_margin_y,
)

box_zmin = max(
    0.0,
    notch_z0 - box_margin_z,
)

box_zmax = Lz

gmsh.model.mesh.field.setNumber(
    box_field,
    "XMin",
    box_xmin,
)

gmsh.model.mesh.field.setNumber(
    box_field,
    "XMax",
    box_xmax,
)

gmsh.model.mesh.field.setNumber(
    box_field,
    "YMin",
    box_ymin,
)

gmsh.model.mesh.field.setNumber(
    box_field,
    "YMax",
    box_ymax,
)

gmsh.model.mesh.field.setNumber(
    box_field,
    "ZMin",
    box_zmin,
)

gmsh.model.mesh.field.setNumber(
    box_field,
    "ZMax",
    box_zmax,
)

# -------------------------------------------------------------------------
# Smooth transition from fine to coarse mesh
# -------------------------------------------------------------------------

gmsh.model.mesh.field.setNumber(
    box_field,
    "Thickness",
    15.0,
)

gmsh.model.mesh.field.setAsBackgroundMesh(
    box_field
)

# =========================================================================
# Additional 3D meshing controls
# =========================================================================

# Use Delaunay tetrahedralization.
gmsh.option.setNumber(
    "Mesh.Algorithm3D",
    1,
)

# Avoid excessively abrupt mesh-size changes.
gmsh.option.setNumber(
    "Mesh.CharacteristicLengthFactor",
    1.0,
)

# =========================================================================
# Generate tetrahedral mesh
# =========================================================================

gmsh.model.mesh.generate(3)

# =========================================================================
# Gmsh built-in mesh optimization
# =========================================================================

gmsh.model.mesh.optimize("Netgen")
# =========================================================================
# Mesh statistics
# =========================================================================

node_tags, node_coords, _ = gmsh.model.mesh.getNodes()

print(
    "Number of mesh nodes:",
    len(node_tags),
)

element_types, element_tags, _ = gmsh.model.mesh.getElements(3)

num_tets = 0

for etype, tags in zip(element_types, element_tags):

    # Gmsh element type 4 = 4-node tetrahedron
    if etype == 4:
        num_tets += len(tags)

print(
    "Number of tetrahedra:",
    num_tets,
)

# =========================================================================
# Write mesh
# =========================================================================

output_file = "02/Lx500_1C_AD.msh"

gmsh.write(output_file)

print("Wrote:", output_file)

# =========================================================================
# Finalize
# =========================================================================

gmsh.finalize()

Crack surface tags: [34, 35, 36, 37]                                                                                                    
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 38 (Line)
Info    : [ 10%] Meshing curve 39 (Line)
Info    : [ 10%] Meshing curve 40 (Line)
Info    : [ 10%] Meshing curve 41 (Line)
Info    : [ 20%] Meshing curve 42 (Line)
Info    : [ 20%] Meshing curve 43 (Line)
Info    : [ 20%] Meshing curve 44 (Line)
Info    : [ 20%] Meshing curve 45 (Line)
Info    : [ 30%] Meshing curve 46 (Line)
Info    : [ 30%] Meshing curve 47 (Line)
Info    : [ 30%] Meshing curve 48 (Line)
Info    : [ 30%] Meshing curve 49 (Line)
Info    : [ 40%] Meshing curve 50 (Line)
Info    : [ 40%] Meshing curve 51 (Line)
Info    : [ 40%] Meshing curve 52 (Line)
Info    : [ 40%] Meshing curve 53 (Line)
Info    : [ 50%] Meshing curve 54 (Line)
Info    : [ 50%] Meshing curve 55 (Line)
Info    : [ 50%] Meshing curve 56 (Line)
Info    : [ 50%] Meshing curve 57 (Line)
Info    : [ 60%] Me

Info    : 0 swaps performed 
Info    : ImproveMesh 
Info    : Total badness = 13629.5 
Info    : Total badness = 13579.2 
Info    : CombineImprove 
Info    : 11 elements combined 
Info    : ImproveMesh 
Info    : Total badness = 13476 
Info    : Total badness = 13470.8 
Info    : SplitImprove 
Info    : badmax = 9.82627 
Info    : 0 splits performed 
Info    : ImproveMesh 
Info    : Total badness = 13470.8 
Info    : Total badness = 13470.6 
Info    : SwapImprove  
Info    : 116 swaps performed 
Info    : SwapImprove2  
Info    : 0 swaps performed 
Info    : ImproveMesh 
Info    : Total badness = 13429.7 
Info    : Total badness = 13401.6 
Info    : Optimizing volume 2
Info    : CalcLocalH: 2483 Points 7082 Elements 4952 Surface Elements 
Info    : Remove Illegal Elements 
Info    : 766 illegal tets 
Info    : SplitImprove 
Info    : badmax = 23.236 
Info    : 63 splits performed 
Info    : SwapImprove  
Info    : 100 swaps performed 
Info    : SwapImprove2  
Info    : 0 swaps performe

In [5]:
import meshio

mesh = meshio.read("02/Lx500_1C_AD.msh")

cells = mesh.get_cells_type("tetra")
points = mesh.points

meshio.write("02/Lx500_1C_AD.xdmf", meshio.Mesh(
    points=points,
    cells={"tetra": cells}))

# Outline

In [6]:
import gmsh

gmsh.initialize()
gmsh.model.add("glacier_terminus")

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 500.0
Ly = 750.0
Lz = 125.0

# Pre-crack / notch dimensions [m]
lx = 5.0
ly = 10.0
lz = 10.0

# -------------------------------------------------------------------------
# Mesh size controls [m]
# -------------------------------------------------------------------------

h_min = 2.0
h_max = 6.15

gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)

# -------------------------------------------------------------------------
# Create 3D glacier volume
# -------------------------------------------------------------------------

glacier = gmsh.model.occ.addBox(
    0.0,
    0.0,
    0.0,
    Lx,
    Ly,
    Lz,
)

# -------------------------------------------------------------------------
# Create notch
# -------------------------------------------------------------------------

notch_x0 = 0.5 * Lx - 0.5 * lx
notch_y0 = 0.0
notch_z0 = Lz - lz

notch = gmsh.model.occ.addBox(
    notch_x0,
    notch_y0,
    notch_z0,
    lx,
    ly,
    lz,
)

# -------------------------------------------------------------------------
# Subtract notch from glacier
# -------------------------------------------------------------------------

domain, _ = gmsh.model.occ.cut(
    [(3, glacier)],
    [(3, notch)],
    removeObject=True,
    removeTool=True,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Get all geometric curves (1D entities)
# -------------------------------------------------------------------------

curves = gmsh.model.getEntities(1)
curve_tags = [tag for dim, tag in curves]

# -------------------------------------------------------------------------
# Physical group for the 1D outline/edges
# -------------------------------------------------------------------------

if curve_tags:
    gmsh.model.addPhysicalGroup(1, curve_tags, 1)
    gmsh.model.setPhysicalName(1, 1, "OUTLINE")

# -------------------------------------------------------------------------
# Generate ONLY a 1D line mesh
# -------------------------------------------------------------------------

gmsh.model.mesh.generate(1)

# -------------------------------------------------------------------------
# Write mesh
# -------------------------------------------------------------------------

gmsh.write("01/Lx500_1C_NA_outline.msh")

gmsh.finalize()

Info    : Meshing 1D...                                                                                                   
Info    : [  0%] Meshing curve 13 (Line)
Info    : [ 10%] Meshing curve 14 (Line)
Info    : [ 10%] Meshing curve 15 (Line)
Info    : [ 20%] Meshing curve 16 (Line)
Info    : [ 20%] Meshing curve 17 (Line)
Info    : [ 30%] Meshing curve 18 (Line)
Info    : [ 30%] Meshing curve 19 (Line)
Info    : [ 30%] Meshing curve 20 (Line)
Info    : [ 40%] Meshing curve 21 (Line)
Info    : [ 40%] Meshing curve 23 (Line)
Info    : [ 50%] Meshing curve 24 (Line)
Info    : [ 50%] Meshing curve 25 (Line)
Info    : [ 60%] Meshing curve 26 (Line)
Info    : [ 60%] Meshing curve 27 (Line)
Info    : [ 60%] Meshing curve 28 (Line)
Info    : [ 70%] Meshing curve 29 (Line)
Info    : [ 70%] Meshing curve 30 (Line)
Info    : [ 80%] Meshing curve 31 (Line)
Info    : [ 80%] Meshing curve 32 (Line)
Info    : [ 80%] Meshing curve 33 (Line)
Info    : [ 90%] Meshing curve 34 (Line)
Info    : [ 90%]

In [7]:
mesh = meshio.read("01/Lx500_1C_NA_outline.msh")

cells = mesh.get_cells_type("line")
points = mesh.points

meshio.write("01/Lx500_1C_NA_outline.xdmf", meshio.Mesh(
    points=points,
    cells={"line": cells}))

meshio.write("02/Lx500_1C_AD_outline.xdmf", meshio.Mesh(
    points=points,
    cells={"line": cells}))

meshio.write("10/Lx500_1C_AD_outline.xdmf", meshio.Mesh(
    points=points,
    cells={"line": cells}))

# A10

In [8]:
import gmsh

gmsh.initialize()

gmsh.model.add("glacier_terminus")

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 500.0
Ly = 750.0
Lz = 125.0

# Pre-crack / notch dimensions [m]
lx = 5.0
ly = 10.0
lz = 10.0

# -------------------------------------------------------------------------
# Mesh size controls [m]
# -------------------------------------------------------------------------

h_min = 2.0
h_max = 1.9

# Refined zone size
h_refined = 2.0

gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)

# -------------------------------------------------------------------------
# Glacier geometry
# -------------------------------------------------------------------------

glacier = gmsh.model.occ.addBox(
    0.0,
    0.0,
    0.0,
    Lx,
    Ly,
    Lz,
)

# -------------------------------------------------------------------------
# Notch
# -------------------------------------------------------------------------

notch_x0 = 0.5 * Lx - 0.5 * lx
notch_y0 = 0.0
notch_z0 = Lz - lz

notch = gmsh.model.occ.addBox(
    notch_x0,
    notch_y0,
    notch_z0,
    lx,
    ly,
    lz,
)

# Subtract notch from glacier
domain, _ = gmsh.model.occ.cut(
    [(3, glacier)],
    [(3, notch)],
    removeObject=True,
    removeTool=True,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Internal cutting planes
#
# These force conforming mesh surfaces and therefore nodes at:
#
# z = Lz / 4 = 31.25 m
# z = Lz / 2 = 62.50 m
# -------------------------------------------------------------------------

z_quarter = Lz / 4.0
z_half = Lz / 2.0

plane_quarter = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    z_quarter,
    Lx,
    Ly,
)

plane_half = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    z_half,
    Lx,
    Ly,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Fragment the glacier with the two internal planes
# -------------------------------------------------------------------------

domain, _ = gmsh.model.occ.fragment(
    domain,
    [
        (2, plane_quarter),
        (2, plane_half),
    ],
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Physical volume
# -------------------------------------------------------------------------

volume_tags = [tag for dim, tag in domain if dim == 3]

gmsh.model.addPhysicalGroup(
    3,
    volume_tags,
    1,
)

gmsh.model.setPhysicalName(
    3,
    1,
    "GLACIER",
)

# -------------------------------------------------------------------------
# Local refinement zone
#
# Crack center:
#       x = 250 m
#
# Refined region:
#       x = 220 -> 280 m
#       y =   0 -> 750 m
#       z =   0 -> 125 m
#
# -------------------------------------------------------------------------

crack_center_x = 0.5 * Lx

refinement_width = 30.0

field_box = gmsh.model.mesh.field.add("Box")

gmsh.model.mesh.field.setNumber(
    field_box,
    "VIn",
    h_refined,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "VOut",
    h_max,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "XMin",
    crack_center_x - refinement_width,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "XMax",
    crack_center_x + refinement_width,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "YMin",
    0.0,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "YMax",
    Ly,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "ZMin",
    0.0,
)

gmsh.model.mesh.field.setNumber(
    field_box,
    "ZMax",
    Lz,
)

# -------------------------------------------------------------------------
# Smooth transition around refined zone
# -------------------------------------------------------------------------

gmsh.model.mesh.field.setNumber(
    field_box,
    "Thickness",
    10.0,
)

gmsh.model.mesh.field.setAsBackgroundMesh(
    field_box
)

# -------------------------------------------------------------------------
# Mesh settings
# -------------------------------------------------------------------------

gmsh.option.setNumber(
    "Mesh.Algorithm3D",
    10,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeFromCurvature",
    0,
)

# Prevent point/curve/surface sizes from overriding background field
gmsh.option.setNumber(
    "Mesh.MeshSizeFromPoints",
    0,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeExtendFromBoundary",
    0,
)

# -------------------------------------------------------------------------
# Generate tetrahedral mesh
# -------------------------------------------------------------------------

gmsh.model.mesh.generate(3)


gmsh.write("10/Lx500_1C_NA.msh")

gmsh.finalize()

Info    : Meshing 1D...                                                                                                                 
Info    : [  0%] Meshing curve 38 (Line)
Info    : [ 10%] Meshing curve 39 (Line)
Info    : [ 10%] Meshing curve 40 (Line)
Info    : [ 10%] Meshing curve 41 (Line)
Info    : [ 20%] Meshing curve 42 (Line)
Info    : [ 20%] Meshing curve 43 (Line)
Info    : [ 20%] Meshing curve 44 (Line)
Info    : [ 20%] Meshing curve 45 (Line)
Info    : [ 30%] Meshing curve 46 (Line)
Info    : [ 30%] Meshing curve 47 (Line)
Info    : [ 30%] Meshing curve 48 (Line)
Info    : [ 30%] Meshing curve 49 (Line)
Info    : [ 40%] Meshing curve 50 (Line)
Info    : [ 40%] Meshing curve 51 (Line)
Info    : [ 40%] Meshing curve 52 (Line)
Info    : [ 40%] Meshing curve 53 (Line)
Info    : [ 50%] Meshing curve 54 (Line)
Info    : [ 50%] Meshing curve 55 (Line)
Info    : [ 50%] Meshing curve 56 (Line)
Info    : [ 50%] Meshing curve 57 (Line)
Info    : [ 60%] Meshing curve 58 (Line)
In

In [9]:
import meshio

mesh = meshio.read("10/Lx500_1C_NA.msh")

cells = mesh.get_cells_type("tetra")
points = mesh.points

meshio.write("10/Lx500_1C_NA.xdmf", meshio.Mesh(
    points=points,
    cells={"tetra": cells}))